# bfeax GPU benchmark — scan vs unrolled ALP recurrence

Compares two implementations of `jax.jit(jax.vmap(grad(Phi)))` on GPU:
- **scan**: `_alp_recurrence` uses `jax.lax.scan` over `l` → **one GPU kernel** for the full recurrence
- **unrolled**: original Python-loop version → **~44 separate XLA ops**, each a potential kernel launch

NFW profile, `l_max=8`, `n_r=128`. Protocol: 2 warmup calls per N, median of 10 timed reps.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

print("JAX version:", jax.__version__)
print("Devices:    ", jax.devices())

In [ ]:
import time
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from bfe import MultipoleExpansion
from bfe.sph_harm import _alp_recurrence  # scan version (current)

## Parameters

In [ ]:
L_MAX  = 8
N_R    = 128
R_MIN  = 0.01
R_MAX  = 300.0
N_REPS = 10

N_vals = [1, 3, 10, 30, 100, 300, 1_000, 3_000, 10_000, 100_000, 1_000_000]

# Agama Multipole CPU reference (laptop, l_max=8, n_r=128)
agama_cpu_N  = [1, 3, 10, 30, 100, 300, 1_000, 3_000, 10_000, 100_000]
agama_cpu_ms = [0.000, 0.001, 0.001, 0.002, 0.007, 0.019, 0.065, 0.169, 0.170, 1.416]

## Unrolled ALP recurrence (baseline)

Monkey-patch the module to use the original Python-loop version so we can time it on the same GPU.

In [ ]:
import bfe.sph_harm as _sh
import math

def _alp_recurrence_unrolled(l_max, x):
    """Original unrolled version — Python loops unroll at trace time."""
    P = [[None] * (l_max + 1) for _ in range(l_max + 1)]
    P[0][0] = jnp.ones_like(x)
    sin_theta = jnp.sqrt(jnp.clip(1.0 - x * x, 0.0))
    for l in range(1, l_max + 1):
        P[l][l]     = -(2*l - 1) * sin_theta * P[l-1][l-1]
        P[l][l - 1] =  (2*l - 1) * x         * P[l-1][l-1]
        for m in range(l - 2, -1, -1):
            P[l][m] = ((2*l-1)*x*P[l-1][m] - (l+m-1)*P[l-2][m]) / (l-m)
    # Return as (l_max+1, l_max+1, *x.shape) to match scan interface
    rows = []
    for l in range(l_max + 1):
        row = jnp.stack([P[l][m] if P[l][m] is not None else jnp.zeros_like(x)
                         for m in range(l_max + 1)])
        rows.append(row)
    return jnp.stack(rows)  # (l_max+1, l_max+1, *x.shape)

print("Unrolled function defined.")

## Build expansion & force functions

In [ ]:
t0 = time.perf_counter()
exp = MultipoleExpansion.from_spheroid(
    rho0=1.0, alpha=1.0, beta=3.0, gamma=1.0, a=1.0,
    r_min=R_MIN, r_max=R_MAX, n_r=N_R, l_max=L_MAX,
)
print(f"Built expansion in {time.perf_counter()-t0:.2f}s")

_grad_phi = jax.grad(exp.__call__, argnums=(0, 1, 2))

# ── Scan version (current bfe code) ─────────────────────────────────────
accel_scan = jax.jit(jax.vmap(_grad_phi))

# ── Unrolled version: temporarily swap _alp_recurrence ──────────────────
_sh._alp_recurrence = _alp_recurrence_unrolled
accel_unrolled = jax.jit(jax.vmap(_grad_phi))
# Restore scan version
from bfe.sph_harm import _alp_recurrence as _alp_scan
_sh._alp_recurrence = _alp_scan

print("Both accel_fn ready (not yet compiled)")

## Timing loop

In [ ]:
rng = np.random.default_rng(0)
times_scan     = []
times_unrolled = []

def bench(fn, x_j, y_j, z_j):
    """2 warmup calls, then N_REPS timed calls. Returns (t_compile, t_warmup2, t_med)."""
    t0 = time.perf_counter()
    out = fn(x_j, y_j, z_j); jax.block_until_ready(out)
    t_compile = time.perf_counter() - t0

    t0 = time.perf_counter()
    out = fn(x_j, y_j, z_j); jax.block_until_ready(out)
    t_warmup2 = time.perf_counter() - t0

    ts = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        out = fn(x_j, y_j, z_j); jax.block_until_ready(out)
        ts.append(time.perf_counter() - t0)
    return t_compile, t_warmup2, np.median(ts)

print(f"{'N':>9}  {'compile scan':>13}  {'compile unroll':>15}  "
      f"{'scan (ms)':>10}  {'unrolled (ms)':>14}  {'speedup':>8}")
print("-" * 80)

for N in N_vals:
    r     = np.exp(rng.uniform(np.log(0.1), np.log(20.0), N))
    cos_t = rng.uniform(-1.0, 1.0, N)
    sin_t = np.sqrt(1.0 - cos_t**2)
    phi_a = rng.uniform(0.0, 2.0 * np.pi, N)
    x_j   = jnp.array(r * sin_t * np.cos(phi_a))
    y_j   = jnp.array(r * sin_t * np.sin(phi_a))
    z_j   = jnp.array(r * cos_t)

    tc_s, tw_s, t_s = bench(accel_scan,     x_j, y_j, z_j)
    tc_u, tw_u, t_u = bench(accel_unrolled, x_j, y_j, z_j)

    times_scan.append(t_s)
    times_unrolled.append(t_u)

    print(f"{N:>9d}  {tc_s*1e3:>12.1f}ms  {tc_u*1e3:>14.1f}ms  "
          f"{t_s*1e3:>10.3f}  {t_u*1e3:>14.3f}  {t_u/t_s:>7.2f}x")

## Plot

In [ ]:
N_arr  = np.array(N_vals)
t_s    = np.array(times_scan)
t_u    = np.array(times_unrolled)
N_ag   = np.array(agama_cpu_N)
t_ag   = np.array(agama_cpu_ms) / 1e3

device_name = str(jax.devices()[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"GPU benchmark — scan vs unrolled ALP recurrence\n"
    f"NFW l_max={L_MAX}, n_r={N_R} | device: {device_name}",
    fontsize=12, fontweight="bold",
)

# ── Left: wall clock time ────────────────────────────────────────────────
ax = axes[0]
ax.loglog(N_arr, t_s * 1e3, "o-",  color="C0", lw=2,   ms=7, label="bfeax scan (GPU)")
ax.loglog(N_arr, t_u * 1e3, "o--", color="C3", lw=1.5, ms=5, label="bfeax unrolled (GPU)")
ax.loglog(N_ag,  t_ag* 1e3, "s--", color="C1", lw=2,   ms=7, label="Agama Multipole (C++ CPU ref)")

ax.set_xlabel("Number of force evaluations $N$", fontsize=11)
ax.set_ylabel("Wall clock time (ms)", fontsize=11)
ax.set_title("Wall clock time vs $N$")
ax.legend(fontsize=10)
ax.grid(True, which="both", alpha=0.3)

# ── Right: throughput ────────────────────────────────────────────────────
ax = axes[1]
ax.loglog(N_arr, N_arr / t_s,  "o-",  color="C0", lw=2,   ms=7, label="bfeax scan (GPU)")
ax.loglog(N_arr, N_arr / t_u,  "o--", color="C3", lw=1.5, ms=5, label="bfeax unrolled (GPU)")
ax.loglog(N_ag,  N_ag  / t_ag, "s--", color="C1", lw=2,   ms=7, label="Agama Multipole (C++ CPU ref)")

ax.set_xlabel("Number of force evaluations $N$", fontsize=11)
ax.set_ylabel("Throughput  (evaluations / second)", fontsize=11)
ax.set_title("Throughput vs $N$")
ax.legend(fontsize=10)
ax.grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.savefig("benchmark_gpu.png", dpi=150)
plt.show()
print("Saved: benchmark_gpu.png")

## Summary table

In [ ]:
print(f"{'N':>9}  {'scan (ms)':>10}  {'unrolled (ms)':>14}  {'speedup':>8}  {'scan M/s':>10}")
print("-" * 58)
for i, N in enumerate(N_vals):
    print(f"{N:>9d}  {times_scan[i]*1e3:>10.3f}  {times_unrolled[i]*1e3:>14.3f}  "
          f"{times_unrolled[i]/times_scan[i]:>7.2f}x  {N/times_scan[i]/1e6:>10.3f}")